# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pangakeerthi11/keerthi-flyrank-ml-internship-2026/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis + time window

**Unit of analysis:** One row represents the observed daily performance of one content item for one pseudonymized client on one reporting date. The proposed record key is `report_date`, `client_hash_id`, and `content_hash_id`.

**Table:** The source is the March 2026 partition of the FlyRank daily content-performance warehouse.

**Time window:** March 1–31, 2026, is my development window.

**Prediction target:** I will investigate whether GSC clicks decrease on the next calendar day compared with the current day. This is a measurable proxy for a next-day click decline, not an explanation of why performance changed.

**Deliberately excluded:** Client and content identifiers are used to identify records, not as model features. Future-day values and any label-derived fields are excluded from the honest feature set to prevent leakage.

In [12]:
import duckdb

con = duckdb.connect()

march_file = file_path

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields: feature / label / context / excluded

**Features — maximum five**

1. `gsc_impressions` — current-day search impressions.
2. `gsc_clicks` — current-day search clicks.
3. `gsc_avg_position` — current-day average search position.
4. `ga4_pageviews` — current-day pageviews.
5. `sessions_organic` — current-day organic sessions.

These features are eligible only when their source data is available at the prediction moment.

**Label / proxy**

`next_day_click_decline` is 1 when the next day's GSC clicks are lower than the current day's clicks, and 0 otherwise. Both days must have GSC data available, and the next observation must be the following calendar day.

**Context**

`report_date`, `month`, `client_hash_id`, and `content_hash_id` identify the observation, reporting period, client, and content item. The source-availability flags describe whether measurements are available. These fields are not model features.

**Excluded**

- Client and content identifiers are excluded from model inputs because the model could memorize identities.
- Future-day metrics are excluded because they are unavailable at the decision moment.
- The target and any column derived from it are excluded from the honest model because they cause label leakage.
- Rows with unavailable GA4 data are excluded from this five-feature experiment. A zero-filled value must not automatically be interpreted as genuine zero activity.

**Prediction moment:** After the current day's measurements are available and before the next day's clicks are known.

In [13]:
# Inspect the available fields and data types
schema = con.execute("""
    DESCRIBE SELECT * FROM read_parquet(?)
""", [march_file]).fetchdf()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verify it with queries

I use March 2026 as my development window. The three verification queries check the proposed record grain, the row count and date span, and source availability. The availability check uses `IS TRUE`; records with unavailable source data are not treated as confirmed measurements. I will use the results to decide whether the proposed grain and time window are supported by the data.

In [14]:
# QUERY 1 — Verify record grain
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (
            report_date,
            client_hash_id,
            content_hash_id
        )) AS distinct_keys,
        COUNT(*) - COUNT(DISTINCT (
            report_date,
            client_hash_id,
            content_hash_id
        )) AS duplicate_key_rows
    FROM read_parquet(?)
""", [march_file]).fetchdf())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_keys,duplicate_key_rows
0,9841378,9841378,0


In [15]:
# QUERY 2 — Verify row count, date span and missing values
display(con.execute("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(*) FILTER (
            WHERE report_date >= DATE '2026-03-01'
              AND report_date < DATE '2026-04-01'
        ) AS rows_in_march,
        COUNT(*) FILTER (
            WHERE gsc_clicks IS NULL
        ) AS missing_gsc_clicks,
        COUNT(*) FILTER (
            WHERE gsc_impressions IS NULL
        ) AS missing_gsc_impressions,
        COUNT(*) FILTER (
            WHERE ga4_pageviews IS NULL
        ) AS missing_ga4_pageviews,
        COUNT(*) FILTER (
            WHERE sessions_organic IS NULL
        ) AS missing_sessions_organic
    FROM read_parquet(?)
""", [march_file]).fetchdf())

,row_count,first_date,last_date,rows_in_march,missing_gsc_clicks,missing_gsc_impressions,missing_ga4_pageviews,missing_sessions_organic
0,9841378,2026-03-01,2026-03-31,9841378,0,0,3018741,3018741


In [16]:
# QUERY 3 — Verify source availability
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
              AND gsc_clicks IS NOT NULL
              AND gsc_impressions IS NOT NULL
              AND gsc_avg_position IS NOT NULL
              AND ga4_pageviews IS NOT NULL
              AND sessions_organic IS NOT NULL
        ) AS rows_with_all_five_features
    FROM read_parquet(?)
""", [march_file]).fetchdf())

,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows,rows_with_all_five_features
0,9841378,3611061,413966,364347,364347


### Five features — available when?

- `gsc_impressions`: Knowable at the decision moment because it is the current day's observed search-impression count and GSC data is available.
- `gsc_clicks`: Knowable at the decision moment because it is the current day's click count, not the next day's clicks used in the label.
- `gsc_avg_position`: Knowable at the decision moment because it measures current-day search position; records with nonpositive position values are excluded.
- `ga4_pageviews`: Knowable at the decision moment because the current GA4 source is marked available.
- `sessions_organic`: Knowable at the decision moment because it is a current-day observed metric and the GA4 source is marked available.

The label is measured on the next calendar day. Records without a consecutive next-day observation or the required source availability are excluded. The feature frame is limited to 20,000 eligible records for this initial experiment.

In [18]:

feature_frame = con.execute("""
    WITH ordered AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            ga4_pageviews,
            sessions_organic,
            gsc_data_available,
            ga4_data_available,
            LEAD(report_date) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
            ) AS next_report_date,
            LEAD(gsc_clicks) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
            ) AS next_gsc_clicks,
            LEAD(gsc_data_available) OVER (
                PARTITION BY client_hash_id, content_hash_id
                ORDER BY report_date
            ) AS next_gsc_available
        FROM read_parquet(?)
    )
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        sessions_organic,
        CASE
            WHEN next_gsc_clicks < gsc_clicks THEN 1
            ELSE 0
        END AS next_day_click_decline
    FROM ordered
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
      AND next_report_date = report_date + INTERVAL '1 day'
      AND next_gsc_available IS TRUE
      AND next_gsc_clicks IS NOT NULL
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
      AND gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
      AND gsc_avg_position > 0
      AND ga4_pageviews IS NOT NULL
      AND sessions_organic IS NOT NULL
    ORDER BY random()
    LIMIT 20000
""", [march_file]).fetchdf()

print("Feature frame shape:", feature_frame.shape)
print("Target distribution:")
display(
    feature_frame["next_day_click_decline"]
    .value_counts()
    .rename_axis("label")
    .to_frame("rows")
)
display(feature_frame.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (20000, 9)
Target distribution:


,rows
label,
0,13426
1,6574


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,sessions_organic,next_day_click_decline
0,2026-03-05,client_23a62021009f63c4,content_4d6c05534e763d2c,71,1,5.183099,2,2,0
1,2026-03-28,client_23a62021009f63c4,content_8f2cf1d35a79e816,7,0,2.428571,2,0,0
2,2026-03-26,client_e5c2aa26a8598242,content_9f04c6cef5f95c3c,10,0,0.100000,1,0,0
3,2026-03-13,client_23a62021009f63c4,content_abf8e0784f34f737,113,0,40.955752,1,0,0
4,2026-03-24,client_23a62021009f63c4,content_86d353e164519f11,185,1,28.540541,3,0,0


### Leakage lesson

I deliberately added `deliberate_label_leak`, which is an exact copy of the target. The model can use this information to predict the answer, producing a misleadingly high score. This feature would not be available at the real prediction moment.

I then removed the leaked feature and evaluated a decision tree using the five permitted features on the same held-out rows. The score without leakage is the honest result to retain. I will report the actual scores produced by the notebook rather than assuming that the honest model must perform well.

This experiment shows why future outcomes and label-derived columns must never be used as model inputs.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

**Named limitation: incomplete GA4 availability.**

In the March data, 413,966 of 9,841,378 rows have GA4 data marked available, while 9,427,412 rows have GA4 marked unavailable. This means the five-feature experiment uses only a restricted subset of the warehouse records.

Unavailable GA4 values must not be interpreted as genuine zero activity. Client histories may also differ, and this analysis cannot prove why clicks decrease or establish causation.

The results are directional decision support, not a guarantee of future performance. The model's scores from one month should not be assumed to generalize to every client or future month.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.